# AutoGluon

In [1]:
import qlib
import pandas as pd
import numpy as np
from qlib.contrib.data.handler import Alpha158
from qlib.data.dataset import DatasetH
from autogluon.tabular import TabularPredictor

In [2]:
# 1. Inicializar Qlib
qlib.init(provider_uri='/home/toni/.qlib/qlib_data/us_data',region=qlib.constant.REG_US)

[56549:MainThread](2026-06-01 14:20:54,357) INFO - qlib.Initialization - [config.py:453] - default_conf: client.
[56549:MainThread](2026-06-01 14:20:54,929) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[56549:MainThread](2026-06-01 14:20:54,930) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/home/toni/.qlib/qlib_data/us_data')}


In [ ]:
import qlib
from qlib.constant import REG_US
from qlib.data import D

qlib.init(provider_uri="/home/toni/.qlib/qlib_data/us_data", region=REG_US)

df = D.features(["^GSPC"], ["$close", "$factor"], start_time="2020-01-01", end_time="2026-05-31", freq="day")

print(df.shape)
print(df.empty)
print(df.tail())


[56549:MainThread](2026-06-01 14:20:54,940) INFO - qlib.Initialization - [config.py:453] - default_conf: client.
[56549:MainThread](2026-06-01 14:20:54,942) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[56549:MainThread](2026-06-01 14:20:54,943) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/home/toni/.qlib/qlib_data/us_data')}


(1591, 2)
False
                         $close   $factor
instrument datetime                      
^GSPC      2026-04-27  4.882702  0.000681
           2026-04-28  4.858805  0.000681
           2026-04-29  4.856866  0.000681
           2026-04-30  4.906591  0.000681
           2026-05-01  4.920960  0.000681


In [4]:
# 2. Cargar dataset Alpha154/158
data_handler_config= {
    "start_time":"2018-01-01",
    "end_time":"2026-05-31",
    "fit_start_time":"2018-01-01",
    "fit_end_time":"2024-12-31",
    "instruments":"sp500",
}

handler= Alpha158(**data_handler_config)

[56549:MainThread](2026-06-01 14:21:07,237) INFO - qlib.timer - [log.py:127] - Time cost: 12.203s | Loading data Done
[56549:MainThread](2026-06-01 14:21:08,152) INFO - qlib.timer - [log.py:127] - Time cost: 0.232s | DropnaLabel Done
[56549:MainThread](2026-06-01 14:21:09,629) INFO - qlib.timer - [log.py:127] - Time cost: 1.476s | CSZScoreNorm Done
[56549:MainThread](2026-06-01 14:21:09,632) INFO - qlib.timer - [log.py:127] - Time cost: 2.394s | fit & process data Done
[56549:MainThread](2026-06-01 14:21:09,633) INFO - qlib.timer - [log.py:127] - Time cost: 14.599s | Init data Done


In [5]:
# 3. Obtener features y labels
features_df= handler.fetch(col_set="feature")
labels_df= handler.fetch(col_set="label")

In [6]:
label_column='LABEL0'

In [7]:
# 4. Combinar en un solo DataFrame
data= pd.concat([features_df, labels_df],axis=1)
# Remove columns that are completely unavailable, e.g. VWAP0 if no $vwap exists
data = data.dropna(axis=1, how="all")

# Make sure label is numeric and finite
data[label_column] = pd.to_numeric(data[label_column], errors="coerce")
data = data.replace([np.inf, -np.inf], np.nan)
data = data[np.isfinite(data[label_column])]

print("Rows after label cleanup:", len(data))
print("Remaining label NaNs:", data[label_column].isna().sum())

Rows after label cleanup: 1037333
Remaining label NaNs: 0


In [8]:
data

KMID      KLEN     KMID2       KUP      KUP2  \
datetime   instrument                                                     
2018-01-02 A           0.002670  0.008158  0.327272  0.004301  0.527267   
           AAL         0.012612  0.022931  0.550001  0.002102  0.091664   
           AAP         0.051437  0.081467  0.631385  0.018236  0.223845   
           AAPL        0.012341  0.017866  0.690787  0.000235  0.013159   
           ABBV        0.013074  0.022133  0.590698  0.005044  0.227907   
...                         ...       ...       ...       ...       ...   
2026-05-27 XYZ         0.021599  0.041982  0.514480  0.016307  0.388416   
           YUM        -0.003605  0.021367 -0.168713  0.014288  0.668708   
           ZBH        -0.026645  0.045980 -0.579487  0.010139  0.220512   
           ZBRA       -0.000910  0.036955 -0.024628  0.020812  0.563168   
           ZTS        -0.006748  0.030117 -0.224063  0.013997  0.464733   

                           KLOW     KLOW2      KSFT     KSFT2     OPEN0  ...  \
datetime   instrument                                                    ...   
2018-01-02 A           0.001187  0.145461 -0.000445 -0.054535  0.997337  ...   
           AAL         0.008217  0.358336  0.018727  0.816673  0.987545  ...   
           AAP         0.011794  0.144769  0.044995  0.552309  0.951079  ...   
           AAPL        0.005289  0.296054  0.017395  0.973681  0.987809  ...   
           ABBV        0.004015  0.181395  0.012044  0.544186  0.987095  ...   
...                         ...       ...       ...       ...       ...  ...   
2026-05-27 XYZ         0.004077  0.097104  0.009369  0.223168  0.978858  ...   
           YUM         0.003474  0.162579 -0.014419 -0.674842  1.003618  ...   
           ZBH         0.009196  0.200000 -0.027588 -0.599999  1.027374  ...   
           ZBRA        0.015233  0.412204 -0.006489 -0.175592  1.000911  ...   
           ZTS         0.009373  0.311204 -0.011372 -0.377593  1.006794  ...   

                        VSUMN10   VSUMN20   VSUMN30   VSUMN60    VSUMD5  \
datetime   instrument                                                     
2018-01-02 A           0.726731  0.586454  0.532266  0.515107 -0.083773   
           AAL         0.634902  0.533584  0.504173  0.500709  0.456964   
           AAP         0.403379  0.482207  0.490406  0.494112  0.388357   
           AAPL        0.608529  0.550233  0.495184  0.494970  0.211619   
           ABBV        0.737471  0.504350  0.497144  0.499255  0.472006   
...                         ...       ...       ...       ...       ...   
2026-05-27 XYZ         0.493372  0.468474  0.494440  0.567612  0.084977   
           YUM         0.453547  0.566095  0.477911  0.497167  0.098398   
           ZBH         0.615164  0.813929  0.520610  0.510146 -0.704191   
           ZBRA        0.850741  0.481492  0.494191  0.498047 -0.136103   
           ZTS         0.794463  0.461786  0.465627  0.477425  0.091637   

                        VSUMD10   VSUMD20   VSUMD30   VSUMD60    LABEL0  
datetime   instrument                                                    
2018-01-02 A          -0.453462 -0.172908 -0.064533 -0.030214 -0.007501  
           AAL        -0.269803 -0.067167 -0.008345 -0.001418  0.006305  
           AAP         0.193242  0.035586  0.019188  0.011776  0.036899  
           AAPL       -0.217058 -0.100466  0.009631  0.010061  0.004645  
           ABBV       -0.474942 -0.008700  0.005711  0.001491 -0.005703  
...                         ...       ...       ...       ...       ...  
2026-05-27 XYZ         0.013256  0.063052  0.011120 -0.135224  0.018426  
           YUM         0.092907 -0.132190  0.044177  0.005665 -0.013864  
           ZBH        -0.230327 -0.627858 -0.041219 -0.020292 -0.005916  
           ZBRA       -0.701483  0.037016  0.011617  0.003906 -0.017225  
           ZTS        -0.588927  0.076428  0.068746  0.045149 -0.007410  

[1037333 rows x 158 columns]

In [9]:
# 5. Preparar train/test split temporal (importante en finanzas!)
dt = data.index.get_level_values("datetime")
train_data = data[dt < "2023-01-01"]
test_data = data[dt >= "2023-01-01"]

train_data = train_data.reset_index()
test_data = test_data.reset_index()

print(f"Train samples:{len(train_data)}, Test samples:{len(test_data)}")
print(f"Features:{features_df.shape[1]}")

Train samples:614648, Test samples:422685
Features:158


In [10]:
# 6. Entrenar con AutoGluon
label_column='LABEL0'# Nombre típico de la columna objetivo en Qlib

predictor= TabularPredictor(
    label=label_column,
    path='qlib_autogluon_models/',
    eval_metric='rmse'# o 'mae' para regresión
).fit(
    train_data=train_data,
    time_limit=3600,# 1 hora
    presets='best_quality',
    verbosity=2
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Tue Nov 5 00:21:55 UTC 2024
CPU Count:          22
Pytorch Version:    2.7.0
CUDA Version:       12.8
GPU Memory:         GPU 0: 15.99/15.99 GB
Total GPU Memory:   Free: 15.99 GB, Allocated: 0.00 GB, Total: 15.99 GB
GPU Count:          1
Memory Avail:       8.57 GB / 15.46 GB (55.4%)
Disk Space Avail:   365.83 GB / 1896.48 GB (19.3%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked over

In [11]:
# 7. Evaluar
test_features= test_data.drop(columns=[label_column])
test_labels= test_data[label_column]

predictions= predictor.predict(test_features)
performance= predictor.evaluate(test_data)

print(f"\n📈 Performance en Test:")
print(performance)

# 8. Leaderboard de modelos
predictor.leaderboard(test_data)


📈 Performance en Test:
{'root_mean_squared_error': np.float32(-0.02069156), 'mean_squared_error': -0.0004281406872905791, 'mean_absolute_error': -0.013933883979916573, 'r2': -0.06058955192565918, 'pearsonr': 0.01755714975297451, 'median_absolute_error': -0.00997381005436182}


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBMXT_BAG_L1,-0.020215,-0.021495,root_mean_squared_error,2.616097,0.631788,1613.585980,2.616097,0.631788,1613.585980,1,True,1
1,WeightedEnsemble_L2,-0.020215,-0.021495,root_mean_squared_error,2.625290,0.638183,1613.594388,0.009193,0.006395,0.008407,2,True,2
2,LightGBMXT_BAG_L2,-0.020692,-0.021146,root_mean_squared_error,4.478236,0.993160,2244.907848,1.862139,0.361372,631.321867,2,True,3
3,WeightedEnsemble_L3,-0.020692,-0.021146,root_mean_squared_error,4.490067,0.998198,2245.066022,0.011831,0.005038,0.158174,3,True,4
